In [107]:
# Librerias Externas
import ee  # Librería de Google Earth Engine
import folium
from shapely.geometry import shape
from itertools import cycle
import requests
# Biblioteca propia
from query_poligonos import extractor_poligonos_localidades
from resampler_poligonos import crear_diccionario_subpoligonos
import matplotlib.pyplot as plt
import geemap

In [5]:
list(range(0,3))

[0, 1, 2]

In [77]:
# Ruta al archivo JSON descargado
SERVICE_ACCOUNT = 'gee-service-account@gee-tesis.iam.gserviceaccount.com'
KEY_PATH = '../../gee-tesis-a0f65918b305.json'

# Autenticación e inicialización de Earth Engine
credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, KEY_PATH)
ee.Initialize(credentials)

print("¡Autenticación exitosa con Google Earth Engine!")

¡Autenticación exitosa con Google Earth Engine!


In [79]:
poligono_cb_sc=extractor_poligonos_localidades(['Ciudad Bolívar','San Cristóbal'])
subpoligonos_cb_sc_diccionario = crear_diccionario_subpoligonos(poligono_cb_sc, tamano_m=600)

## Visualizacion Comprobacion

In [94]:
# Crear un mapa centrado en Bogotá, 1 km hacia el sur
m = folium.Map(location=[4.6 - 0.05, -74.1], zoom_start=13)

# Lista de colores para asignar dinámicamente a las localidades
colores = ['blue', 'green', 'red', 'orange', 'purple', 'brown', 'pink', 'gray']
colores_ciclo = cycle(colores)  # Ciclo infinito de colores

# Crear un diccionario de colores asignados dinámicamente a cada localidad
colores_localidades = {localidad: next(colores_ciclo) for localidad in poligono_cb_sc.keys()}

# Función para definir estilos en función de la localidad
def get_style_function(localidad):
    color = colores_localidades.get(localidad, 'black')  # Color predeterminado: negro si no se encuentra
    return lambda x: {'color': color, 'fillOpacity': 0.5}

# Agregar las localidades originales al mapa
for localidad, geojson in poligono_cb_sc.items():
    folium.GeoJson(
        data=geojson,
        name=localidad,
        style_function=get_style_function(localidad)  # Llamar a la función de estilo
    ).add_to(m)

# Agregar los subpolígonos al mapa
for localidad, subpoligonos in subpoligonos_cb_sc_diccionario.items():
    for subpoligono_nombre, subpoligono_geojson in subpoligonos.items():
        folium.GeoJson(
            data=subpoligono_geojson,
            style_function=lambda x: {'color': 'black', 'fillOpacity': 0},  # Cuadraditos en negro sin relleno
            popup=folium.Popup(subpoligono_nombre)
        ).add_to(m)

# Guardar el mapa interactivo
m

In [82]:
subpoligonos_cb_sc_diccionario['Ciudad Bolívar'].keys()

dict_keys(['subpoligono_1', 'subpoligono_2', 'subpoligono_3', 'subpoligono_4', 'subpoligono_5', 'subpoligono_6', 'subpoligono_7', 'subpoligono_8', 'subpoligono_9', 'subpoligono_10', 'subpoligono_11', 'subpoligono_12', 'subpoligono_13', 'subpoligono_14', 'subpoligono_15', 'subpoligono_16', 'subpoligono_17', 'subpoligono_18', 'subpoligono_19', 'subpoligono_20', 'subpoligono_21', 'subpoligono_22', 'subpoligono_23', 'subpoligono_24', 'subpoligono_25', 'subpoligono_26', 'subpoligono_27', 'subpoligono_28', 'subpoligono_29', 'subpoligono_30', 'subpoligono_31', 'subpoligono_32', 'subpoligono_33', 'subpoligono_34', 'subpoligono_35', 'subpoligono_36', 'subpoligono_37', 'subpoligono_38', 'subpoligono_39', 'subpoligono_40', 'subpoligono_41', 'subpoligono_42', 'subpoligono_43', 'subpoligono_44', 'subpoligono_45', 'subpoligono_46', 'subpoligono_47', 'subpoligono_48', 'subpoligono_49', 'subpoligono_50', 'subpoligono_51', 'subpoligono_52', 'subpoligono_53', 'subpoligono_54', 'subpoligono_55', 'subpoli

In [101]:
subpoligono_1=subpoligonos_cb_sc_diccionario['Ciudad Bolívar']['subpoligono_306']

In [102]:
subpoligono_1

{'type': 'Polygon',
 'coordinates': (((-74.15344566318264, 4.530172223723281),
   (-74.1534393474662, 4.5355996750577585),
   (-74.14803107148286, 4.535593312867788),
   (-74.14803742753556, 4.5301658691770905),
   (-74.15344566318264, 4.530172223723281)),)}

In [103]:
import simplekml
# Crear el archivo KML
kml = simplekml.Kml()

# Extraer las coordenadas del GeoJSON
coordinates = subpoligono_1['coordinates'][0]

# Crear el polígono en el archivo KML
pol = kml.newpolygon(name="Mi Polígono")
pol.outerboundaryis = coordinates
pol.style.polystyle.color = simplekml.Color.changealphaint(100, simplekml.Color.green)  # Verde transparente
pol.style.polystyle.fill = True
pol.style.polystyle.outline = True

# Guardar el archivo KML
kml.save("mi_poligono.kml")
print("Archivo KML creado: 'mi_poligono.kml'")

Archivo KML creado: 'mi_poligono.kml'


In [104]:
# Convertir el polígono a ee.Geometry
geometry = ee.Geometry(subpoligono_1)

# Verificar el polígono en la consola
print(geometry.getInfo())

{'type': 'Polygon', 'coordinates': [[[-74.15344566318264, 4.530172223723281], [-74.14803742753556, 4.5301658691770905], [-74.14803107148286, 4.535593312867788], [-74.1534393474662, 4.5355996750577585], [-74.15344566318264, 4.530172223723281]]]}


In [116]:
# Filtrar imágenes Sentinel-2 para el año 2022
image_collection = ee.ImageCollection("COPERNICUS/S2_HARMONIZED") \
    .filterBounds(geometry) \
    .filterDate('2022-01-01', '2022-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .sort('CLOUDY_PIXEL_PERCENTAGE')

# Lista de imágenes filtradas
images = image_collection.toList(image_collection.size())

# Parámetros de visualización RGB
rgb_vis_params = {
    'min': 0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2']  # Bandas RGB
}

# Ajustar resolución y zoom al descargar imágenes
scale = 10  # Resolución en metros/píxel
image_size = 2048  # Tamaño en píxeles de la imagen final

In [123]:
import math

# Parámetros del círculo y los cuadrados
circle_radius = 1000  # Radio del círculo en metros
square_side = 100  # Lado del cuadrado en metros

# Cálculo del número de anillos posibles
num_rings = circle_radius // (square_side / 2)  # Cada anillo está a una distancia igual al radio del cuadrado

# Calcular el número de cuadrados por anillo
total_squares = 0
for ring in range(1, int(num_rings) + 1):
    # Radio del anillo actual
    ring_radius = ring * (square_side / 2)
    # Perímetro del anillo
    ring_perimeter = 2 * math.pi * ring_radius
    # Número de cuadrados en el anillo
    squares_in_ring = ring_perimeter // square_side
    total_squares += int(squares_in_ring)

print(f"El número máximo de cuadrados que pueden acomodarse es: {total_squares}")

El número máximo de cuadrados que pueden acomodarse es: 649


In [118]:

# Descargar cada imagen con mejor calidad
for i in range(min(5, images.size().getInfo())):  # Limitar a las 5 mejores imágenes
    # Obtener la imagen i-ésima
    image = ee.Image(images.get(i))
    image_id = image.get('system:index').getInfo()

    # Generar la URL del thumbnail
    url = image.getThumbURL({
        'region': geometry,
        'dimensions': image_size,  # Tamaño de la imagen en píxeles
        'format': 'png',
        'min': rgb_vis_params['min'],
        'max': rgb_vis_params['max'],
        'scale': scale,  # Resolución ajustada
        'bands': ','.join(rgb_vis_params['bands'])
    })

    # Descargar la imagen
    response = requests.get(url)
    if response.status_code == 200:
        img_data = BytesIO(response.content)
        img = Image.open(img_data)

        # Guardar la imagen como PNG
        filename = f"image_{image_id}.png"
        img.save(filename, "PNG")
        print(f"Imagen descargada y guardada como {filename}.")
    else:
        print(f"Error al descargar la imagen {image_id}: {response.status_code}")


EEException: Image.clipToBoundsAndScale: "scale" may not be specified along with "width", "height", or "maxDimension".